# 02 (full data) · e5 embeddings and candidate buckets for every train and test record

The same method as `02_e5_embeddings.ipynb`, run on **all** records instead of a sample:
* text `name_norm | addr_norm` from 01's `processed/` files, encoded with
  `intfloat/multilingual-e5-small` (MIT), in fp16 on both GPUs. Names and addresses written in an
  Indian script are transliterated to Latin letters from the original text instead (see below).
* per country, every S1 gets its `TOP_K` most similar S2/S3 records (its **bucket**), found by
  exact search on mean-centered embeddings, on both GPUs

Both splits are processed:
* **train** buckets are what the full LightGBM notebook learns from
* **test** buckets are what it predicts on, and they become `candidate_pairs.tsv`

Only the buckets are saved. The embeddings of all 24M records would not fit in Kaggle's 20 GB
output limit.

**Input:**
* 01's `processed/` folder, from a run with `FULL_RUN=true`. 01 now also writes `test_s2_*` and
  `test_s3_*`, which the test buckets need.
* `train_ground_truth.tsv`, used only for the bucket-recall report.

**Output** (`embeddings_full/multilingual-e5-small/`):

| File | Content |
|---|---|
| `buckets/<split>_<country>.parquet` | one row per (S1, candidate): `s1_entity_id`, `rank`, `candidate_entity_id`, `candidate_source`, `score` (centered cosine), `cosine` (original e5 cosine), `country` |
| `bucket_recall_train.csv` | share of true matches inside the train buckets, and the best score a perfect matcher could reach on them |
| `manifest.json` | settings, record counts and timings |

**Run on Kaggle:**
1. Accelerator: GPU T4 x2. Internet on, to download the model (or attach it and set `E5_MODEL`).
2. Attach the competition data and the output of 01 (`FULL_RUN=true`), then Run all.
3. About 24M texts are encoded. My estimate is roughly an hour on two T4s, plus a few minutes of
   search per country. A bucket file that already exists is skipped, so a re-run in the same
   session continues where it stopped.

Settings are in the first code cell and can also be set as environment variables: `PROCESSED_DIR`,
`DATA_ROOT`, `EMB_FULL_DIR`, `SPLITS`, `TOP_K`, `E5_MODEL`, `TEXT_MODE`, `COUNTRIES`.

In [ ]:
import csv
import gc
import glob
import inspect
import json
import os
import re
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import torch

SPLITS = [s.strip() for s in os.environ.get("SPLITS", "train,test").split(",") if s.strip()]
TOP_K = int(os.environ.get("TOP_K", 30))                       # candidates kept per S1 (the matcher uses all 30)
TEXT_MODE = os.environ.get("TEXT_MODE", "fixed")               # fixed | processed | latin | raw (see "Build the buckets")
COUNTRIES = [c for c in os.environ.get("COUNTRIES", "").split(",") if c]   # empty = every country (testing aid)
MODEL_NAME = os.environ.get("E5_MODEL", "intfloat/multilingual-e5-small")
PREFIX = "query: "           # E5 expects "query: " for symmetric similarity
MAX_SEQ_LENGTH = 128         # tokens; names + addresses are short
ENCODE_CHUNK = 1_000_000     # texts per encoding call; keeps memory flat for countries with millions of records
assert TEXT_MODE in {"fixed", "processed", "latin", "raw"}, TEXT_MODE
assert set(SPLITS) <= {"train", "test"}, SPLITS


def find_processed_dir():
    """Folder with 01's per-country parquet: $PROCESSED_DIR, ./processed, /kaggle/working, any Kaggle input."""
    candidates = [Path(os.environ["PROCESSED_DIR"])] if os.environ.get("PROCESSED_DIR") else []
    candidates += [Path.cwd() / "processed", Path("/kaggle/working/processed")]
    candidates += sorted({Path(p).parent for p in glob.glob("/kaggle/input/**/train_s1_*.parquet", recursive=True)})
    for c in candidates:
        if c.is_dir() and any(c.glob("train_s1_*.parquet")):
            return c
    raise FileNotFoundError("processed/ not found: run 01_eda_preprocessing.ipynb with FULL_RUN=true or set PROCESSED_DIR")


def find_ground_truth():
    """train_ground_truth.tsv under $DATA_ROOT, /kaggle/input or the local dataset folder; None if absent."""
    roots = [os.environ.get("DATA_ROOT", ""), "/kaggle/input", str(Path.cwd().parent / "Data" / "student_resource" / "dataset")]
    for root in roots:
        if root and os.path.isdir(root):
            hits = [h for h in glob.glob(os.path.join(root, "**", "train_ground_truth.tsv"), recursive=True)
                    if "__MACOSX" not in h]
            if hits:
                return Path(sorted(hits, key=len)[0])
    return None


PROCESSED_DIR = find_processed_dir()
GT_FILE = find_ground_truth()
OUT_DIR = Path(os.environ.get("EMB_FULL_DIR") or ("/kaggle/working/embeddings_full" if os.path.isdir("/kaggle/working")
                                                  else "embeddings_full")) / MODEL_NAME.rstrip("/").split("/")[-1]
BUCKET_DIR = OUT_DIR / "buckets"
BUCKET_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
GPUS = [f"cuda:{i}" for i in range(torch.cuda.device_count())] if DEVICE == "cuda" else []   # all used (T4 x2 on Kaggle)
if os.environ.get("GPU_LIST"):                                  # testing aid, e.g. "cuda:0,cuda:0"
    GPUS = os.environ["GPU_LIST"].split(",")
GPU_NAME = torch.cuda.get_device_name(0) if DEVICE == "cuda" else "cpu"
# fp16 speeds up tensor-core GPUs (T4, A100) but is ~4x slower on GTX 16xx cards (measured on a GTX 1650).
USE_FP16 = DEVICE == "cuda" and "GTX 16" not in GPU_NAME
BATCH_SIZE = 1024 if USE_FP16 else 512                         # texts per GPU batch when encoding
CHUNK_SIZE = 16 * BATCH_SIZE                                   # texts per work unit handed to one GPU process
SEARCH_DTYPE = torch.float16 if USE_FP16 else torch.float32    # first scoring pass (fp16 = tensor cores)
RESCORE_EXTRA = 32                                             # the top k + 32 of that pass are re-scored in fp32
GPU_MEMORY = torch.cuda.get_device_properties(0).total_memory if DEVICE == "cuda" else 0
print(f"processed: {PROCESSED_DIR}\nground truth: {GT_FILE}\noutput: {OUT_DIR}")
print(f"device: {len(GPUS)} x {GPU_NAME} | fp16: {USE_FP16} | batch: {BATCH_SIZE} | splits: {SPLITS} | top-k: {TOP_K} | "
      f"text mode: {TEXT_MODE}")

## Inventory of `processed/`
Every (split, country) with S1 records gets buckets when S2/S3 records exist for it. If
`test_s2_*` / `test_s3_*` are missing, 01 ran with the old file list: re-run it with `FULL_RUN=true`.

In [ ]:
FILE_RE = re.compile(r"^(train|test)_(s[123])_(.+)\.parquet$")
INVENTORY = pd.DataFrame([{"split": m.group(1), "source": m.group(2), "country": m.group(3), "path": f,
                           "rows": pq.ParquetFile(f).metadata.num_rows}
                          for f in sorted(PROCESSED_DIR.glob("*.parquet")) if (m := FILE_RE.match(f.name))])
assert len(INVENTORY), f"no <split>_<source>_<country>.parquet files in {PROCESSED_DIR}"
if (INVENTORY["rows"] <= 40_000).all():
    print("WARNING: every file has <= 40,000 rows: processed/ looks like 01's sample mode. Run 01 with FULL_RUN=true.")
for split in SPLITS:
    if not ((INVENTORY["split"] == split) & INVENTORY["source"].isin(["s2", "s3"])).any():
        print(f"WARNING: no {split}_s2_* / {split}_s3_* files, so no {split} buckets can be built. "
              "Re-run 01 (FULL_RUN=true) with the updated file list.")
INVENTORY.pivot_table(index=["split", "country"], columns="source", values="rows", aggfunc="sum").fillna(0).astype(int)

## Encoding on the GPUs
* Each unique text is encoded once, in fp16 on T4s, `BATCH_SIZE` texts per batch.
* Texts are sorted by length, so each batch holds texts of similar length (little padding).
* With several GPUs, one encoding process runs per GPU, and work units of `CHUNK_SIZE` texts go
  to whichever GPU is free.
* Texts are encoded `ENCODE_CHUNK` at a time and stored as fp16 straight away, which keeps memory
  flat even for countries with millions of records.

In [ ]:
from sentence_transformers import SentenceTransformer

_model, _pool = None, None


def get_model():
    """Load the e5 model once (kept on the CPU when it is copied to several GPUs)."""
    global _model
    if _model is None:
        _model = SentenceTransformer(MODEL_NAME, device="cpu" if len(GPUS) > 1 else DEVICE)
        _model.max_seq_length = MAX_SEQ_LENGTH
        if USE_FP16:
            _model.half()
    return _model


def get_pool():
    """One encoding process per GPU, started once and reused for every file."""
    global _pool
    if _pool is None:
        _pool = get_model().start_multi_process_pool(target_devices=GPUS)
    return _pool


def stop_pool():
    """Stop the encoding processes."""
    global _pool
    if _pool is not None:
        SentenceTransformer.stop_multi_process_pool(_pool)
        _pool = None


def encode(batch):
    """float32 embeddings of a list of texts, on all GPUs."""
    model = get_model()
    if len(GPUS) > 1:
        if "pool" in inspect.signature(model.encode).parameters:          # sentence-transformers >= 5
            return model.encode(batch, pool=get_pool(), batch_size=BATCH_SIZE, chunk_size=CHUNK_SIZE,
                                convert_to_numpy=True, show_progress_bar=False)
        return model.encode_multi_process(batch, get_pool(), batch_size=BATCH_SIZE, chunk_size=CHUNK_SIZE)
    return model.encode(batch, batch_size=BATCH_SIZE, convert_to_numpy=True, show_progress_bar=False)


def embed_all(texts, label):
    """Unit-norm float16 embeddings aligned with texts (each unique text encoded once, longest first)."""
    codes, uniq = pd.factorize(np.asarray(texts, dtype=object))
    order = np.argsort(-np.fromiter((len(t) for t in uniq), np.int64, len(uniq)), kind="stable")
    out, t0 = None, time.time()
    for s in range(0, len(order), ENCODE_CHUNK):
        idx = order[s:s + ENCODE_CHUNK]
        vecs = np.asarray(encode([PREFIX + t for t in uniq[idx]]), dtype=np.float32)
        vecs /= np.maximum(np.linalg.norm(vecs, axis=1, keepdims=True), 1e-12)
        if out is None:
            out = np.empty((len(uniq), vecs.shape[1]), np.float16)
        out[idx] = vecs.astype(np.float16)
        done = min(s + ENCODE_CHUNK, len(order))
        print(f"  {label}: {done:,}/{len(uniq):,} unique texts, {done / max(time.time() - t0, 1e-9):,.0f} texts/s")
    return out[codes]

## Exact search on the GPUs
* Each GPU centers the S2/S3 embeddings (subtracts their mean and re-normalizes, as in notebook
  02), keeps a copy, and takes its share of the S1 query blocks.
* A block is scored against all S2/S3 vectors in one matrix product (fp16 on T4s). Its best
  k + 32 per S1 are re-scored in fp32, and the top k are kept.
* The original e5 cosine of every bucket pair is then computed on the GPU as well.
* Without a GPU, the same search runs with NumPy on the CPU (slow; meant for small tests).

In [ ]:
def _normalize(x):
    return x / x.norm(dim=1, keepdim=True).clamp_min(1e-12)


def exact_search(base16, queries16, mu, k):
    """Top-k rows of base16 for every row of queries16, by inner product of mean-centered, re-normalized vectors."""
    n, d = base16.shape
    scores = np.empty((len(queries16), k), np.float32)
    hits = np.empty((len(queries16), k), np.int64)
    if not GPUS:
        base = base16.astype(np.float32) - mu
        base /= np.maximum(np.linalg.norm(base, axis=1, keepdims=True), 1e-12)
        for s in range(0, len(queries16), 1024):
            q = queries16[s:s + 1024].astype(np.float32) - mu
            q /= np.maximum(np.linalg.norm(q, axis=1, keepdims=True), 1e-12)
            sims = q @ base.T
            top = np.argpartition(-sims, k - 1, axis=1)[:, :k]
            order = np.argsort(-np.take_along_axis(sims, top, axis=1), axis=1)
            hits[s:s + 1024] = np.take_along_axis(top, order, axis=1)
            scores[s:s + 1024] = np.take_along_axis(sims, hits[s:s + 1024], axis=1)
        return scores, hits
    shortlist = min(n, k + RESCORE_EXTRA)
    elem = 2 if SEARCH_DTYPE == torch.float16 else 4
    rows = int(np.clip((GPU_MEMORY // 8) // (n * elem), 64, 4096))    # query rows per block: score block <= 1/8 of GPU memory
    blocks = [(s, min(s + rows, len(queries16))) for s in range(0, len(queries16), rows)]

    def run(device, my_blocks):
        with torch.cuda.device(device):
            mu_t = torch.from_numpy(mu).to(device)
            base = torch.empty((n, d), dtype=SEARCH_DTYPE, device=device)
            for s in range(0, n, 1_000_000):                           # center in chunks (no fp32 copy of everything)
                chunk = torch.from_numpy(base16[s:s + 1_000_000]).to(device).float() - mu_t
                base[s:s + len(chunk)] = _normalize(chunk).to(SEARCH_DTYPE)
            del chunk
            for s, e in my_blocks:
                q = _normalize(torch.from_numpy(queries16[s:e]).to(device).float() - mu_t)
                cand = torch.topk(q.to(SEARCH_DTYPE) @ base.T, shortlist, dim=1).indices
                rescored = torch.einsum("qd,qkd->qk", q, base[cand].float())
                top = torch.topk(rescored, k, dim=1)
                scores[s:e] = top.values.cpu().numpy()
                hits[s:e] = torch.gather(cand, 1, top.indices).cpu().numpy()
            del base
            torch.cuda.empty_cache()

    with ThreadPoolExecutor(len(GPUS)) as ex:
        list(ex.map(run, GPUS, [blocks[i::len(GPUS)] for i in range(len(GPUS))]))
    return scores, hits


def pair_cosine(a, a_rows, b, b_rows, chunk=500_000):
    """Dot product of a[a_rows[j]] and b[b_rows[j]] for every j (original e5 cosine), in chunks, on a GPU when there is one."""
    out = np.empty(len(a_rows), np.float32)
    if GPUS:
        with torch.cuda.device(GPUS[0]):
            ta, tb = torch.from_numpy(a).to(GPUS[0]), torch.from_numpy(b).to(GPUS[0])
            for s in range(0, len(a_rows), chunk):
                ia = torch.from_numpy(a_rows[s:s + chunk]).to(GPUS[0])
                ib = torch.from_numpy(b_rows[s:s + chunk]).to(GPUS[0])
                out[s:s + chunk] = (ta[ia].float() * tb[ib].float()).sum(dim=1).cpu().numpy()
            del ta, tb
            torch.cuda.empty_cache()
        return out
    for s in range(0, len(a_rows), chunk):
        out[s:s + chunk] = np.einsum("ij,ij->i", a[a_rows[s:s + chunk]].astype(np.float32),
                                     b[b_rows[s:s + chunk]].astype(np.float32))
    return out

## Ground truth (train buckets report only)
The labels are used only to report how many true matches the train buckets contain. They do not
influence the embeddings or the search.

In [ ]:
OWNER, N_TRUE_ALL = None, None
if "train" in SPLITS and GT_FILE is not None:
    gt = pd.read_csv(GT_FILE, sep="\t", dtype=object, keep_default_na=False, na_filter=False, quoting=csv.QUOTE_NONE)
    N_TRUE_ALL = gt.set_index("source1_entity_id")["matched_entity_ids"].map(lambda s: len(s.split(",")) if s else 0)
    pairs = gt[gt["matched_entity_ids"] != ""]
    pairs = pairs.assign(entity_id=pairs["matched_entity_ids"].str.split(",")).explode("entity_id")
    OWNER = pd.Series(pairs["source1_entity_id"].to_numpy(), index=pairs["entity_id"].to_numpy())
    del gt, pairs
    print(f"{len(OWNER):,} train S2/S3 records have a true S1; {int((N_TRUE_ALL == 0).sum()):,} train S1 are singletons")
else:
    print("no ground truth (or train not in SPLITS): the bucket-recall report is skipped")


def bucket_report(frame, s1_ids, country):
    """Train only: share of true matches in the top-k buckets and the best macro F0.5 a perfect matcher could reach."""
    pos = pd.Series(np.arange(len(s1_ids)), index=s1_ids)
    n_true = N_TRUE_ALL.reindex(s1_ids).fillna(0).to_numpy()
    is_true = frame["candidate_entity_id"].map(OWNER).to_numpy() == frame["s1_entity_id"].to_numpy()
    rows = []
    for k in (5, 10, 20, 30, 50):
        if k > TOP_K:
            continue
        sel = is_true & (frame["rank"].to_numpy() < k)
        found = np.bincount(pos.loc[frame["s1_entity_id"].to_numpy()[sel]].to_numpy(), minlength=len(s1_ids))
        with np.errstate(divide="ignore", invalid="ignore"):
            r = found / n_true
            f = np.where(n_true == 0, 1.0, np.where(found > 0, 1.25 * r / (0.25 + r), 0.0))
        rows.append({"country": country, "k": k, "S1": len(s1_ids),
                     "true matches in bucket": round(float(found.sum() / max(n_true.sum(), 1)), 4),
                     "ceiling macro F0.5": round(float(f.mean()), 4)})
    return rows

## Build the buckets
One (split, country) at a time: read the texts, encode S1 and S2+S3, search, save, free memory.

**Text given to e5** (`TEXT_MODE`, default `fixed`):
* Latin-script records: `name_norm | addr_norm` from 01, unchanged.
* Names or addresses with a native script (Hindi, Telugu, Malayalam, Tamil, ...): transliterated to
  Latin letters from the original text with `anyascii` (ISC license), e.g. "లోటస్ కేర్" -> "lots ker".
  01's cleaning strips combining marks, which deletes the vowel signs of Indian scripts, so these
  names reached e5 garbled. On a dense regional test bed, 57% of the true matches with an
  Indian-script name were missing from the top-20 buckets (4.7% for the other matches).

In [ ]:
TEXT_COLS = {"fixed": ["name_norm", "addr_norm", "business_name", "business_address"],
             "processed": ["name_norm", "addr_norm"], "latin": ["name_lat", "legal_form", "addr_norm"],
             "raw": ["business_name", "business_address"]}[TEXT_MODE]
if TEXT_MODE == "fixed":
    import importlib.util
    import subprocess
    import sys
    if importlib.util.find_spec("anyascii") is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "anyascii"])
    from anyascii import anyascii
# a text has a native script when it has a character outside Latin, Latin-1/Extended and general punctuation
_LATIN_RANGES = [(0x00, 0x7F), (0xA0, 0x24F), (0x1E00, 0x1EFF), (0x2000, 0x206F), (0x20A0, 0x20CF)]
RE_NATIVE = re.compile("[^" + "".join(re.escape(chr(a)) + "-" + re.escape(chr(b)) for a, b in _LATIN_RANGES) + "]")


def latin(raw, cleaned):
    """cleaned (01's text) for Latin-script text, else the transliterated original."""
    return " ".join(anyascii(raw).lower().split()) if RE_NATIVE.search(raw) else cleaned


def read_texts(paths):
    """entity_ids and e5 texts (TEXT_MODE, as in notebook 02) of the records in the given processed files."""
    df = pd.concat([pd.read_parquet(p, columns=["entity_id"] + TEXT_COLS) for p in paths], ignore_index=True)
    for c in TEXT_COLS:
        df[c] = df[c].fillna("").astype(str)
    if TEXT_MODE == "raw":
        names, addrs = df["business_name"], df["business_address"]
    elif TEXT_MODE == "latin":
        names, addrs = (df["name_lat"] + " " + df["legal_form"]).str.strip(), df["addr_norm"]
    elif TEXT_MODE == "fixed":
        names = [latin(r, c) for r, c in zip(df["business_name"], df["name_norm"])]
        addrs = [latin(r, c) for r, c in zip(df["business_address"], df["addr_norm"])]
    else:
        names, addrs = df["name_norm"], df["addr_norm"]
    texts = [f"{n.strip()} | {a.strip()}" if a.strip() else n.strip() for n, a in zip(names, addrs)]
    return df["entity_id"].astype(str).to_numpy(dtype=object), texts


def files(split, sources, country):
    sel = INVENTORY[(INVENTORY["split"] == split) & INVENTORY["source"].isin(sources) & (INVENTORY["country"] == country)]
    return sel.sort_values("source")["path"].tolist()


manifest_path = OUT_DIR / "manifest.json"
MANIFEST = json.loads(manifest_path.read_text(encoding="utf-8")) if manifest_path.exists() else {}
SETTINGS = {"model": MODEL_NAME, "prefix": PREFIX, "max_seq_length": MAX_SEQ_LENGTH, "text_mode": TEXT_MODE,
            "top_k": TOP_K, "processed_dir": str(PROCESSED_DIR)}
if MANIFEST.get("settings") != SETTINGS:
    MANIFEST = {"settings": SETTINGS, "done": {}, "recall": []}
T_ALL = time.time()
for split in SPLITS:
    for country in sorted(INVENTORY.loc[(INVENTORY["split"] == split) & (INVENTORY["source"] == "s1"), "country"].unique()):
        if COUNTRIES and country not in COUNTRIES:
            continue
        key, dst = f"{split}_{country}", BUCKET_DIR / f"{split}_{country}.parquet"
        if key in MANIFEST["done"] and dst.exists():
            print(f"{key}: already built, skipped")
            continue
        cand_paths = files(split, ["s2", "s3"], country)
        if not cand_paths:
            print(f"{key}: no S2/S3 records, no buckets (its S1 get empty candidate lists)")
            continue
        t0 = time.time()
        s1_ids, s1_texts = read_texts(files(split, ["s1"], country))
        c_ids, c_texts = read_texts(cand_paths)
        if not len(c_ids) or not len(s1_ids):
            print(f"{key}: no records to search, skipped")
            continue
        print(f"{key}: {len(s1_ids):,} S1 and {len(c_ids):,} S2/S3 records")
        s1_emb = embed_all(s1_texts, f"{key} S1")
        c_emb = embed_all(c_texts, f"{key} S2/S3")
        del s1_texts, c_texts
        t_embed = time.time() - t0

        t1 = time.time()
        mu = c_emb.mean(axis=0, dtype=np.float64).astype(np.float32)[None, :]
        k = min(TOP_K, len(c_ids))
        scores, hits = exact_search(c_emb, s1_emb, mu, k)
        qi, ci = np.repeat(np.arange(len(s1_ids)), k), hits.ravel()
        frame = pd.DataFrame({"s1_entity_id": s1_ids[qi], "rank": np.tile(np.arange(k, dtype=np.int16), len(s1_ids)),
                              "candidate_entity_id": c_ids[ci]})
        frame["candidate_source"] = frame["candidate_entity_id"].str[:2]
        frame["score"] = scores.ravel()
        frame["cosine"] = pair_cosine(s1_emb, qi, c_emb, ci)
        frame["country"] = country
        t_search = time.time() - t1
        frame.to_parquet(dst, index=False)

        if split == "train" and OWNER is not None:
            MANIFEST["recall"] += bucket_report(frame, s1_ids, country)
        MANIFEST["done"][key] = {"s1": len(s1_ids), "s2_s3": len(c_ids), "pairs": len(frame),
                                 "embed_s": round(t_embed), "search_s": round(t_search)}
        manifest_path.write_text(json.dumps(MANIFEST, indent=1), encoding="utf-8")
        print(f"{key}: saved {len(frame):,} pairs | encode {t_embed / 60:.1f} min, search {t_search / 60:.1f} min")
        del s1_ids, c_ids, s1_emb, c_emb, scores, hits, qi, ci, frame
        gc.collect()
        if GPUS:
            torch.cuda.empty_cache()
stop_pool()
print(f"all buckets done in {(time.time() - T_ALL) / 60:.1f} min")
pd.DataFrame([{"bucket": k, **v} for k, v in MANIFEST["done"].items()])

## How many true matches do the train buckets hold?
For train S1 (all of them, at full scale): the share of their true matches that are inside the
top-k bucket, and the macro F0.5 a perfect matcher would reach using only those buckets. The
same numbers are expected for test, which is built the same way.

In [ ]:
RECALL = pd.DataFrame(MANIFEST["recall"])
if len(RECALL):
    RECALL.to_csv(OUT_DIR / "bucket_recall_train.csv", index=False)
    display(RECALL.pivot(index="k", columns="country", values=["true matches in bucket", "ceiling macro F0.5"]))
print("bucket files:", sorted(p.name for p in BUCKET_DIR.glob("*.parquet")))